<a href="https://colab.research.google.com/github/Soheilp86/Statistical-Machine-Learning/blob/main/Lab2_Random_Forest.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab:  Selecting features with Random Forests


* **Part 1: Feature selection.** Which features drive house prices, and how sure can you be?


* **Part 2: Filling missing values.** Can a random forest guess missing entries better than simple methods?

**Instructions:** Read each section and write your code in the empty code cells below it. Some cells are already filled in (marked **GIVEN**); just run them. Write your written answers in the marked markdown cells.

In [1]:
# GIVEN: libraries for the whole worksheet
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import fetch_openml, fetch_california_housing
from sklearn.model_selection import (train_test_split, cross_val_score,
                                     KFold, RepeatedStratifiedKFold)
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.feature_selection import SelectFromModel
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error

from sklearn.experimental import enable_iterative_imputer  # needed before importing IterativeImputer
from sklearn.impute import SimpleImputer, KNNImputer, IterativeImputer

## Part 1: Feature Selection on House Prices

We use the **Ames Housing** dataset: 1,460 houses sold in Ames, Iowa, with about 80 features each (size, quality, year built, garage, basement, ...). The target is `SalePrice` in dollars.

In [2]:
#Load the Ames Housing dataset from sklearn and save it as a panda dataframe. The target variable is SalePrice. Call it y and use X for other features.
# Load the data (ask gemni:)
ames = fetch_openml(name="house_prices", as_frame=True)
df = ames.frame

To keep things simple, keep only the **numeric** features.

In [ ]:
# disect the dataframe to keep only numerical features

Let's briefly explore the data. I'm not asking for a detailed EDA, enough to feel comfortable playing with it.

In [ ]:
#Short EDA

Do train-test split before preprocessing. A few numeric columns have missing values (e.g. LotFrontage). For both train and test dataset, fill them with the median. Make sure you keep them separate so no information from the test set is used.

In [ ]:
#train-test split

To check whether our feature-selection method can tell real signal from junk, we also add **5 noise columns** that have nothing to do with price:

* `noise_1`, `noise_2`, `noise_3`: random numbers between 0 and 1 (continuous)
* `noise_4`, `noise_5`: random 0/1 values (binary). We add these columns to both train and test.

A good feature selection method should rank all five noise columns near the bottom.

In [ ]:
rng = np.random.RandomState(42)
for i in [1, 2, 3]:
    X[f"noise_{i}"] = rng.rand(len(X))             # continuous noise
for i in [4, 5]:
    X[f"noise_{i}"] = rng.randint(0, 2, len(X))    # binary noise


noise_cols = [f"noise_{i}" for i in range(1, 6)]
print("Features:", X.shape[1], "  Train:", len(X_train), "  Test:", len(X_test))

In [3]:
# GIVEN: load the data, add noise columns, split, fill missing values
ames = fetch_openml(name="house_prices", as_frame=True)
df = ames.frame

y = df["SalePrice"].astype(float)
X = df.drop(columns=["SalePrice", "Id"], errors="ignore").select_dtypes(include="number")

rng = np.random.RandomState(42)
for i in [1, 2, 3]:
    X[f"noise_{i}"] = rng.rand(len(X))             # continuous noise
for i in [4, 5]:
    X[f"noise_{i}"] = rng.randint(0, 2, len(X))    # binary noise

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

# A few numeric columns have missing values (e.g. LotFrontage). Fill them with the
# TRAINING median, so no information from the test set is used.
medians = X_train.median()
X_train = X_train.fillna(medians)
X_test = X_test.fillna(medians)

noise_cols = [f"noise_{i}" for i in range(1, 6)]
print("Features:", X.shape[1], "  Train:", len(X_train), "  Test:", len(X_test))

Features: 41   Train: 1095   Test: 365


### **1. Impurity-based importance**

* Fit a `RandomForestRegressor` on the training set. Use `n_estimators=300`, `max_features=0.33` (about $p/3$), `random_state=0`, and `n_jobs=-1`.
* Report the test RMSE.
* Plot the 20 most important features using `rf.feature_importances_`.
* Print the **rank** of each noise column (1 = most important).

In [ ]:
# Fit the random forest and report the test RMSE

In [ ]:
# Plot the top 20 features by impurity importance, and print the rank of each noise column

**Report your observation here**


### **2. Permutation importance**

* Compute permutation importance on the **test set** with `permutation_importance(rf, X_test, y_test, n_repeats=10, random_state=0, n_jobs=-1)`.


__Note__ If a feature matters, permuting its values should impact the ranking. And if a feature is not useful, permuting it should not have an impact on ranking.


* Print the rank of each noise column again.


* Make a table with both rankings side by side. Find **two features** whose rank changed a lot.

In [ ]:
# Compute permutation importance on the test set and print the rank of the noise columns

In [ ]:
# Compare the two rankings side by side and find features whose rank changed a lot

**Answer:**

* Which method (1 or 2) would you trust more, and why?



* Name two features whose rank changed a lot. Suggest a reason.

*Write your answer here.*

### 3. Keep the top 10 features

* Using the **impurity importance from Task 1** (training data only), keep the 10 most important features.


* Compare **5-fold cross-validated RMSE** on the training set for (a) all features and (b) the top 10. Use `cross_val_score(..., scoring="neg_root_mean_squared_error")` and `KFold(5, shuffle=True, random_state=0)`.


* Also compare the test RMSE of the two models.



__If you dont remember why this step matters, discuss that in your group and ask AI. Compare your answers with AI's!__



In [ ]:
# Select the top 10 features and compare CV RMSE: all features vs. top 10

In [ ]:
# Compare test RMSE: all features vs. top 10

**Report your observation:**

* How much accuracy did you lose (or gain) by keeping only 10 of the features?

*Write your answer here.*

### 4. Is the ranking stable?

If a different sample of houses gives a different top 10, you should not trust your top 10 too much.

* Repeat this 10 times: draw a **bootstrap sample** of the training rows, fit the same forest, and record its top 10 features (by impurity importance).
* Count how many of the 10 runs each feature appears in the top 10.

In [ ]:
# Bootstrap the training set 10 times and count how often each feature is in the top 10

**Your answers:**

* Which features are in the top 10 every time? Which ones only sometimes?
* If your manager asks for "the 10 most important features", what would you tell them about the features near the cutoff?

*Write your answer here.*

### 5. Correlated features

`GrLivArea` (above-ground living area) is strongly correlated with other size features, such as `TotRmsAbvGrd` (total rooms), `1stFlrSF`, and `2ndFlrSF`.

* Print the correlation between `GrLivArea` and these three features.
* Drop `GrLivArea`, refit the forest, and compare the importance of these three features **before and after**.

In [ ]:
# Correlation of GrLivArea with related size features

In [ ]:
# Drop GrLivArea, refit, and compare importance before and after

**Report your observation:**

* What happened to the importance of the related features when `GrLivArea` was removed?


* How much did the test RMSE change? What does this tell you about reading importance for correlated features?

*Write your answer here.*

### 6. Was your cross-validation honest?

In Task 3 you chose the top 10 using the **whole training set**, then ran cross-validation on that same training set. So each validation fold had already "voted" on which features to keep.

A fair check repeats the selection **inside** each fold. Build a `Pipeline` with two steps:

1. `SelectFromModel(RandomForestRegressor(...), max_features=10, threshold=-np.inf)`: keeps the 10 most important features, refit in every fold.
2. A `RandomForestRegressor` that is trained on those 10 features.

Compute its 5-fold CV RMSE with the same `cv` and compare it with the top-10 CV RMSE from Task 3.

In [ ]:
# Build the pipeline and compute its honest CV RMSE

**Your answers:**

* Which CV estimate is more honest, and why?
* Here the difference may be small. When would choosing features *outside* the CV loop give a badly over-optimistic score? (Hint: think of 10,000 features and 100 rows.)

*Write your answer here.*

### Reflection

Your manager asks: *"Which 10 features matter for house price?"* Using your results from Tasks 1–6, write 3–4 sentences: what you would report, how confident you are, and what caveats you would mention.

*Write your answer here.*